In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import seaborn as sns


AOD Dataset 

In [ ]:
folder_path = r"C:\Users\Harsh\OneDrive\Desktop\pollution\aod" 
files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".h5")]
files.sort()  

aod_data_list = []
time_list = []

In [ ]:
# Loop over files
for file in files:
    with h5py.File(file, 'r') as f:
        print("Reading:", file)
        print("Available keys:", list(f.keys()))  # check variable names inside

        # Read variables
        aod = f['AOD'][:]               # (time, lat, lon)
        lat = f['latitude'][:]          # 1D
        lon = f['longitude'][:]         # 1D
        time = f['time'][:]             # 1D or scalar
        
        aod_data_list.append(aod)
        time_list.append(time)

In [ ]:
# Stack all files into one array
aod_array = np.concatenate(aod_data_list, axis=0)   # shape: (T, lat, lon)
time_array = np.concatenate(time_list, axis=0)      # shape: (T,)


In [ ]:
print("AOD array shape:", aod_array.shape)
print("Lat shape:", lat.shape, "Lon shape:", lon.shape)
print("Time shape:", time_array.shape)

Getting Delhi Data From AOD 

In [ ]:
import xarray as xr
aod_da = xr.DataArray(
    aod_array,
    coords={
        "time": time_array,   # we will convert later if needed
        "lat": lat,
        "lon": lon
    },
    dims=["time", "lat", "lon"],
    name="AOD"
)

In [ ]:
print(time_array[:10])

In [ ]:
aod_da = aod_da.where(aod_da > 0)  # remove negative/fill values

In [ ]:
ref_date = pd.to_datetime("2000-01-01")
decoded_time = ref_date + pd.to_timedelta(time_array, unit="m")
aod_da = aod_da.assign_coords(time=decoded_time)

In [ ]:
aod_da = aod_da.sel(time=aod_da['time'].dt.minute == 30)

In [ ]:
print(aod_da.time[:5])

In [ ]:
delhi_mean = aod_da.sel(lat=slice(28.9, 28.4), lon=slice(76.9, 77.4)).mean(dim=["lat","lon"])
print(delhi_mean[:10])

In [ ]:
delhi_mean_df = delhi_mean.to_dataframe().reset_index()

# Ensure datetime type
delhi_mean_df['time'] = pd.to_datetime(delhi_mean_df['time'])

# Set 'time' as index
delhi_mean_df = delhi_mean_df.set_index('time')

# Step 1: Fill with daily mean
delhi_mean_df['AOD'] = delhi_mean_df.groupby(delhi_mean_df.index.date)['AOD'].transform(
    lambda x: x.fillna(x.mean())
)

# Step 2: Interpolate across time
delhi_mean_df['AOD'] = delhi_mean_df['AOD'].interpolate(method='time')

# Step 3: Forward/backward fill if any NaNs remain
delhi_mean_df['AOD'] = delhi_mean_df['AOD'].fillna(method='ffill').fillna(method='bfill')

# Reset index if you want 'time' back as a column
delhi_mean_df = delhi_mean_df.reset_index()




In [ ]:
print("NaN count:", delhi_mean_df['AOD'].isna().sum())

In [ ]:
delhi_mean_df.to_csv("Delhi_AOD_clean.csv", index=False)

In [ ]:
daily_aod = delhi_mean_df.resample('D', on='time')['AOD'].mean()

plt.figure(figsize=(12,6))
plt.plot(daily_aod.index, daily_aod.values, color='green', label='Daily Mean AOD')
plt.title("Delhi Daily Mean AOD", fontsize=14)
plt.xlabel("Date")
plt.ylabel("AOD")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Mean AOD across time
aod_mean = aod_da.mean(dim="time")
aod_mean.plot(cmap="viridis")

# Draw India's bounding box
plt.plot([68, 97, 97, 68, 68],
         [6, 6, 37, 37, 6],
         color="red", linewidth=2)

plt.title("Dataset coverage with India bounding box")
plt.show()


In [ ]:
# Create figure and axis
fig, ax = plt.subplots(figsize=(6, 5))

# Initial plot (first frame)
im = ax.imshow(
    aod_da.isel(time=0), 
    cmap="viridis", 
    origin="lower", 
    extent=[
        float(aod_da.lon.min()), float(aod_da.lon.max()),
        float(aod_da.lat.min()), float(aod_da.lat.max())
    ]
)
ax.set_title(str(aod_da.time.values[0]))
plt.colorbar(im, ax=ax, label="AOD")

# Update function for animation
def update(frame):
    data = aod_da.isel(time=frame)
    im.set_data(data)
    ax.set_title(str(aod_da.time.values[frame]))
    return [im]

# Create animation (looping through every 3rd timestep)
ani = animation.FuncAnimation(
    fig, update, 
    frames=range(0, len(aod_da.time), 3), 
    interval=300, blit=False
)

plt.close(fig)  # Prevent duplicate static plot in notebook

# Show in Jupyter Notebook
from IPython.display import HTML
HTML(ani.to_jshtml())


CPCB DATASET

In [ ]:
cpcb = pd.read_excel(r"C:\Users\Harsh\OneDrive\Desktop\pollution\cpcb1.xlsx")

In [ ]:
cpcb.head()

In [ ]:
cpcb.tail()

In [ ]:
cpcb.isnull().sum()

In [ ]:
cpcb.dropna()

In [ ]:
pm25_mean = cpcb['PM2.5'].mean()
pm25_mean = round(pm25_mean, 2) 
print("Average PM2.5:", pm25_mean)

In [ ]:
cpcb['PM2.5'] = cpcb['PM2.5'].fillna(pm25_mean)


In [ ]:
print(cpcb['PM2.5'].isna().sum()) 


Extracting cpcb data from 5 30 gmt to 8 30 gmt

convert from date to datetime

In [ ]:

cpcb.columns = cpcb.columns.str.strip()  # remove extra spaces

# Convert 'From Date' to datetime
cpcb['From Date'] = pd.to_datetime(cpcb['From Date'], format='%d-%m-%Y %H:%M')


In [ ]:
cpcb.set_index('From Date', inplace=True)


In [ ]:
print(cpcb.columns)


In [ ]:
# Keep only PM2.5 column
cpcb_pm = cpcb[['PM2.5']]

# Resample to 30-minute intervals and interpolate
cpcb_60min = cpcb_pm.resample('60min').interpolate(method='linear')

# Reset index if you want
cpcb_60min = cpcb_60min.reset_index()


In [ ]:
cpcb_60min['Hour'] = cpcb_60min['From Date'].dt.hour
cpcb_filtered = cpcb_60min[(cpcb_60min['Hour'] >= 11) & (cpcb_60min['Hour'] <= 14)].reset_index(drop=True)

print(cpcb_filtered.head(10))


In [ ]:
cpcb_filtered_df = cpcb_filtered.reset_index()
cpcb_filtered_df.to_csv("cpcb.csv", index=False)


In [ ]:
plt.figure(figsize=(12,6))
sns.scatterplot(data=cpcb_60min, x="From Date", y="PM2.5", s=30, color="blue", alpha=0.6)

# Formatting
plt.title("CPCB PM2.5 Scatterplot (60-min interval)", fontsize=14)
plt.xlabel("Time", fontsize=12)
plt.ylabel("PM2.5 (µg/m³)", fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.5)

plt.show()


Merra 2 Dataset

Parameters we use 
1. North 10m
2. East 10m
3. Sea Level Pressure
4. Surface Skin Temperature
5. Total Precipitate Water Vapour

In [ ]:
merra = pd.read_csv("C:\\Users\\Harsh\\OneDrive\\Desktop\\pollution\\merr2.csv")

In [ ]:
merra.head()

In [ ]:
merra.tail()

In [ ]:
merra.info()

In [ ]:
merra.shape

In [ ]:
merra.describe

Line plots (Time Sries Trends) of Sea level pressure, Surface skin temperature, TQV

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,6))

ax1.plot(merra['time'], merra['Sea level pressue'], color="blue", label="Sea level pressure")
ax1.set_ylabel("Pressure (hPa)", color="blue")
ax1.tick_params(axis='y', labelcolor="blue")

ax2 = ax1.twinx()
ax2.plot(merra['time'], merra['Surface skin temp'], color="orange", label="Surface skin temp")
ax2.plot(merra['time'], merra['TQV'], color="green", label="TQV")
ax2.set_ylabel("Temp (K) / TQV", color="green")
ax2.tick_params(axis='y', labelcolor="green")

fig.suptitle("MERRA-2 Variables Over Time")
fig.tight_layout()
plt.show()


Wind vectors (east vs north wind)

In [ ]:
# Subset to avoid clutter
subset = merra.iloc[::50]   # every 50th row

# Create x = time index, y = 0 (all arrows on one line)
x = np.arange(len(subset))
y = np.zeros(len(subset))

u = subset['east wind 10m']
v = subset['north wind 10m']

plt.figure(figsize=(12,4))
plt.quiver(x, y, u, v, angles='xy', scale=20, color='blue')

plt.xlabel("Time Index (subset)")
plt.ylabel("Wind Vector")
plt.title("Wind Vectors at 10m (MERRA-2)")
plt.grid(True)
plt.show()


Wind Rose Plot(east vs north wind)

In [ ]:

# Calculate wind speed and direction
u = merra['east wind 10m'].values
v = merra['north wind 10m'].values

speed = np.sqrt(u**2 + v**2)
direction = (np.arctan2(u, v) * 180 / np.pi) % 360  # wind direction in degrees

# Create bins for direction (like compass)
bins = np.arange(0, 361, 30)  # every 30°
hist, _ = np.histogram(direction, bins=bins)

# Plot wind rose (polar bar plot)
theta = np.deg2rad(bins[:-1])  # convert bins to radians
width = np.deg2rad(30)         # each bar is 30°
r = hist / hist.max()          # normalize

plt.figure(figsize=(8,8))
ax = plt.subplot(111, polar=True)
bars = ax.bar(theta, r, width=width, bottom=0.0, color='blue', alpha=0.6, edgecolor='black')

ax.set_theta_zero_location("N")   # 0° at north
ax.set_theta_direction(-1)        # clockwise
ax.set_title("Wind Rose (MERRA-2 10m Winds)", va='bottom')

plt.show()


Heat map Correlation Between Merra-2 Variables

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(merra.drop(columns=['time']).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between MERRA-2 Variables")
plt.show()


Surface Skin Temperature Vs Water Vapour

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(merra['Surface skin temp'], merra['TQV'], alpha=0.5)

plt.xlabel("Surface Skin Temp (K)")
ax1.set_ylabel("TQV", color="blue")
ax1.set_xlabel("Surface skin temp", color="green")
plt.ylabel("TQV (kg/m^2)")
plt.title("Relationship between Temperature and Water Vapor")
plt.show()


MErging Aod, CPCB and Merra-2 dataset

In [ ]:
merge = pd.read_csv(r"C:\Users\Harsh\OneDrive\Desktop\pollution\merged.csv")

In [ ]:
merge.head()

In [ ]:
merge.tail()

In [ ]:
merge.describe()

In [ ]:
merge.info()

In [ ]:
merge.isnull().sum()

## XGBoost Regression Model for PM2.5 Prediction
Using `merged.csv` to train an XGBoost Regressor model to predict ground-level `PM2.5` from satellite AOD and MERRA-2 meteorological features (`Sea level pressue`, `east wind 10m`, `north wind 10m`, `Surface skin temp`, `TQV`).

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load merged dataset
merge = pd.read_csv("merged.csv")

# Features and Target Variable
X = merge.drop(columns=["time", "PM2.5"])
y = merge["PM2.5"]

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dataset Split Complete:")
print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Testing set:  {X_test.shape[0]} samples, {X_test.shape[1]} features")


In [ ]:
# Initialize and train XGBoost Regressor model
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)
print("XGBoost Regressor model trained successfully!")


In [ ]:
# Evaluate predictions and calculate R^2 and RMSE
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("=" * 45)
print("       XGBoost Model Performance Metrics")
print("=" * 45)
print(f"Train R^2 Score : {train_r2:.4f}")
print(f"Test R^2 Score  : {test_r2:.4f}")
print("-" * 45)
print(f"Train RMSE      : {train_rmse:.4f}")
print(f"Test RMSE       : {test_rmse:.4f}")
print("=" * 45)


In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("Feature Importances:")
print(feature_importance.to_string(index=False))
